In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display
from statsmodels.stats.anova import anova_lm

from analysis.src.data import complete_case_mask
from analysis.src.io import experiment_paths, load_data
from analysis.src.plotting import configure_plotting
from analysis.src.constants import COVARIATES, OUTCOMES

In [ ]:
configure_plotting()

In [ ]:
OUT = experiment_paths('associations/dose_response')
DISEASES = OUTCOMES

data = load_data()

In [ ]:
# Create quintiles and fit quintile models for all QOF outcomes
data['walkability_quintile'] = pd.qcut(data['walkability_score'], q = 5, labels = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])
quintile_rows = []

for outcome in DISEASES:
    model_data = data.loc[complete_case_mask(data, outcome, COVARIATES)]
    
    formula = f"{outcome} ~ C(walkability_quintile, Treatment(reference = 'Q1')) + " + ' + '.join(COVARIATES)
    model = smf.ols(formula, data = model_data).fit()

    for quintile in ['Q2', 'Q3', 'Q4', 'Q5']:
        parameter = f"C(walkability_quintile, Treatment(reference='Q1'))[T.{quintile}]"
        ci = model.conf_int().loc[parameter]
        
        quintile_rows.append(
            {
                'outcome': outcome,
                'quintile': quintile,
                'coefficient': model.params[parameter],
                'ci_lower': ci.iloc[0],
                'ci_upper': ci.iloc[1],
                'p_value': model.pvalues[parameter]
            }
        )

quintile_results = pd.DataFrame(quintile_rows)
quintile_results.to_csv(OUT.quintile_contrasts_csv, index = False)

display(quintile_results.round(4))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize = (16, 8), constrained_layout = True)

for ax, outcome in zip(axes.flat, DISEASES):
    plot_data = quintile_results.query('outcome == @outcome')
    ax.errorbar(
        plot_data.quintile,
        plot_data.coefficient,
        yerr = [plot_data.coefficient - plot_data.ci_lower, plot_data.ci_upper - plot_data.coefficient],
        fmt = '-o',
        color = 'purple',
        capsize = 3,
    )
    ax.axhline(0, color = 'red', linestyle = '--', linewidth = 1)
    ax.set(title = outcome.removeprefix('qof_'), ylabel = 'Difference from Q1')

fig.savefig(OUT.quintile_contrasts_png, dpi = 300, bbox_inches = 'tight')

plt.show()

In [ ]:
# Linear vs. spline tests and marginal predictions for all QOF outcomes
min_w = data['walkability_score'].quantile(0.01)
max_w = data['walkability_score'].quantile(0.99)
w_range = np.linspace(min_w, max_w, 100)

nonlinearity_rows = []
prediction_rows = []
for outcome in DISEASES:
    model_data = data.loc[complete_case_mask(data, outcome, COVARIATES)]
    
    linear_model = smf.ols(
        f'{outcome} ~ walkability_score + ' + ' + '.join(COVARIATES), data = model_data
    ).fit()
    spline_model = smf.ols(
        f'{outcome} ~ cr(walkability_score, df = 4) + ' + ' + '.join(COVARIATES), data = model_data
    ).fit()
    spline_test = anova_lm(linear_model, spline_model)
    
    nonlinearity_rows.append(
        {
            'outcome': outcome,
            'n_lsoas': int(spline_model.nobs),
            'linear_aic': linear_model.aic,
            'spline_aic': spline_model.aic,
            'delta_aic': spline_model.aic - linear_model.aic,
            'nonlinearity_f_statistic': spline_test.iloc[1]['F'],
            'nonlinearity_p_value': spline_test.iloc[1]['Pr(>F)']
        }
    )

    prediction_grid = pd.DataFrame({'walkability_score': w_range})

    for covariate in COVARIATES:
        prediction_grid[covariate] = model_data[covariate].mean()

    predictions = spline_model.get_prediction(prediction_grid).summary_frame(alpha = 0.05)
    prediction_rows.extend(
        {
            'outcome': outcome,
            'walkability_score': score,
            'predicted_prevalence': prediction,
            'ci_lower': lower,
            'ci_upper': upper
        }
        for score, prediction, lower, upper in zip(
            w_range, predictions['mean'], predictions['mean_ci_lower'], predictions['mean_ci_upper']
        )
    )

nonlinearity_results = pd.DataFrame(nonlinearity_rows)
nonlinearity_results.to_csv(OUT.spline_tests_csv, index = False)

display(nonlinearity_results.round(4))

In [ ]:
prediction_results = pd.DataFrame(prediction_rows)
fig, axes = plt.subplots(2, 4, figsize = (16, 8), constrained_layout = True)

for ax, outcome in zip(axes.flat, DISEASES):
    plot_data = prediction_results.query('outcome == @outcome')
    ax.plot(plot_data.walkability_score, plot_data.predicted_prevalence, color = 'teal')
    ax.fill_between(plot_data.walkability_score, plot_data.ci_lower, plot_data.ci_upper, color = 'teal', alpha = 0.2)
    ax.set(title = outcome.removeprefix('qof_'), xlabel = 'Walkability score', ylabel = 'Predicted prevalence')

fig.savefig(OUT.spline_predictions_png, dpi = 300, bbox_inches = 'tight')

plt.show()

In [ ]:
pd.DataFrame(prediction_rows).to_csv(OUT.spline_predictions_csv, index = False)